In [5]:
import pandas as pd

df = pd.read_csv("../data/train_custom.csv")

print(len(df))
df.head()

13158


,tweet_id,sentiment,text
0,39361,positive,Fake news Muslim man was not beaten up in Kari...
1,19737,negative,@ VazeIndian @ RahulGandhi Muslim vo kutte ki ...
2,12664,negative,@ margallass Aray nhe Nawaz sharif sala dulla ...
3,27674,negative,RT @ shaik _ hussam EVM ki rigging nahi hui Hi...
4,18543,negative,@ bprerna Prerna bakshi tum jsise log ho .. ji...


In [ ]:
import openai
import time
import os
import pandas as pd
from concurrent.futures import ThreadPoolExecutor, as_completed

client = openai.OpenAI(api_key="KEY_API")

EMOTION_PROMPT = """You are an emotion classifier for code-mixed Hinglish social media text.
Classify the following tweet into exactly one of these emotions: anger, joy, sadness, trust, fear, surprise.
Respond with only the emotion word, nothing else.

Tweet: {tweet}
Emotion:"""

def label_emotion(row):
    tweet_id, text = row
    for attempt in range(3):  # 3 retries on rate limit
        try:
            response = client.chat.completions.create(
                model="gpt-4o-mini",
                messages=[{"role": "user", "content": EMOTION_PROMPT.format(tweet=text)}],
                temperature=0,
                max_tokens=10
            )
            label = response.choices[0].message.content.strip().lower()
            return tweet_id, label
        except openai.RateLimitError:
            time.sleep(2 ** attempt)  # 1s, 2s, 4s backoff
        except Exception as e:
            return tweet_id, "error"
    return tweet_id, "error"

In [10]:
SAVE_PATH = "../data/emotion_train_labeled.csv"

if os.path.exists(SAVE_PATH):
    already_done = pd.read_csv(SAVE_PATH)
    done_ids = set(already_done['tweet_id'].tolist())
    print(f"Resuming — {len(done_ids)} already labeled")
else:
    already_done = pd.DataFrame()
    done_ids = set()

remaining = df[~df['tweet_id'].isin(done_ids)][['tweet_id', 'text']]
print(f"Tweets remaining: {len(remaining)}")

rows = list(remaining.itertuples(index=False, name=None))
results = []
BATCH_SIZE = 200  # 200 per batch

for i in range(0, len(rows), BATCH_SIZE):
    batch = rows[i:i+BATCH_SIZE]
    
    with ThreadPoolExecutor(max_workers=5) as executor:
        futures = {executor.submit(label_emotion, row): row for row in batch}
        for future in as_completed(futures):
            tweet_id, label = future.result()
            results.append({'tweet_id': tweet_id, 'label': label})
    
    # Save after every batch
    partial = pd.DataFrame(results)
    combined = pd.concat([already_done, partial], ignore_index=True)
    combined.to_csv(SAVE_PATH, index=False)
    
    print(f"Done: {min(i + BATCH_SIZE, len(rows))} / {len(rows)}")
    time.sleep(3)  # 3 second gap between batches

print("✓ Labeling complete!")

Resuming — 4400 already labeled
Tweets remaining: 8758


KeyboardInterrupt: 

In [8]:
done = pd.read_csv("../data/emotion_train_labeled.csv")
print(f"Saved so far: {len(done)}")

Saved so far: 4400


In [11]:
response = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=[{"role": "user", "content": "Say the word: joy"}],
    temperature=0,
    max_tokens=10
)
print(response.choices[0].message.content)

RateLimitError: Error code: 429 - {'error': {'message': 'Rate limit reached for gpt-4o-mini in organization org-hiV86l554TdHLwwdzNFr2Pnt on requests per day (RPD): Limit 10000, Used 10000, Requested 1. Please try again in 8.64s. Visit https://platform.openai.com/account/rate-limits to learn more.', 'type': 'requests', 'param': None, 'code': 'rate_limit_exceeded'}}

In [15]:
import pandas as pd

df1 = pd.read_csv("../data/emotion_labeled_2.csv")
df2 = pd.read_csv("../data/emotion_labeled_1.csv")

print("df1 columns:", df1.columns.tolist())
print("df2 columns:", df2.columns.tolist())
print("\ndf1 shape:", df1.shape)
print("df2 shape:", df2.shape)

df1 columns: ['tweet_id', 'label']
df2 columns: ['tweet_id', 'sentiment', 'text', 'emotion']

df1 shape: (4400, 2)
df2 shape: (2825, 4)


In [16]:
# Get text back for df1
train_original = pd.read_csv("../data/train_custom.csv")
df1 = df1.merge(train_original[['tweet_id', 'text']], on='tweet_id', how='left')
df1 = df1.rename(columns={'label': 'emotion'})

# Keep only what we need from df2
df2 = df2[['tweet_id', 'text', 'emotion']]

# Merge
merged = pd.concat([df1, df2], ignore_index=True)
print(merged.columns.tolist())
print(merged.shape)
print(merged['emotion'].value_counts())

['tweet_id', 'emotion', 'text']
(7225, 3)
emotion
anger        3182
joy          2635
sadness       560
trust         392
error         324
surprise      113
fear           18
confusion       1
Name: count, dtype: int64


In [25]:
from sklearn.model_selection import train_test_split

df1 = pd.read_csv("../data/emotion_labeled_1.csv")
df2 = pd.read_csv("../data/emotion_labeled_2.csv")

# Merge and deduplicate - keep df1 version if duplicate
merged = pd.concat([df1, df2], ignore_index=True)
merged = merged.drop_duplicates(subset='tweet_id', keep='first')
print(f"After dedup: {len(merged)}")

# Clean
keep = ['anger', 'joy', 'sadness', 'trust']
merged_clean = merged[merged['emotion'].isin(keep)].reset_index(drop=True)
print(merged_clean['emotion'].value_counts())
print(f"Total: {len(merged_clean)}")

# Split
train, dev = train_test_split(merged_clean, test_size=0.15,
                               stratify=merged_clean['emotion'],
                               random_state=42)

# Verify zero overlap
overlap = pd.merge(train, dev, on='tweet_id')
print(f"\nOverlap after fix: {len(overlap)}")
print(f"Train: {len(train)}, Dev: {len(dev)}")

After dedup: 6357
emotion
anger      1350
joy        1092
sadness     244
trust       139
Name: count, dtype: int64
Total: 2825

Overlap after fix: 0
Train: 2401, Dev: 424


In [22]:
import pandas as pd

train = pd.read_csv("../data/emotion_train_final.csv")
dev = pd.read_csv("../data/emotion_dev_final.csv")

# Check for overlap
overlap = pd.merge(train, dev, on='tweet_id')
print(f"Train size: {len(train)}")
print(f"Dev size: {len(dev)}")
print(f"Overlapping tweet_ids: {len(overlap)}")

Train size: 5753
Dev size: 1016
Overlapping tweet_ids: 190


In [21]:
train_custom = pd.read_csv("../data/train_custom.csv")
dev_custom = pd.read_csv("../data/dev_custom.csv")

overlap = pd.merge(train_custom, dev_custom, on='tweet_id')
print(f"train_custom size: {len(train_custom)}")
print(f"dev_custom size: {len(dev_custom)}")
print(f"Overlapping tweet_ids: {len(overlap)}")

train_custom size: 13158
dev_custom size: 2322
Overlapping tweet_ids: 0


In [23]:
print("Duplicate tweet_ids in train:", train['tweet_id'].duplicated().sum())
print("Duplicate tweet_ids in dev:", dev['tweet_id'].duplicated().sum())

Duplicate tweet_ids in train: 580
Duplicate tweet_ids in dev: 19


In [24]:
df1 = pd.read_csv("../data/emotion_labeled_1.csv")
df2 = pd.read_csv("../data/emotion_labeled_2.csv")

print("df1 duplicate tweet_ids:", df1['tweet_id'].duplicated().sum())
print("df2 duplicate tweet_ids:", df2['tweet_id'].duplicated().sum())

overlap = pd.merge(df1, df2, on='tweet_id')
print(f"Overlap between df1 and df2: {len(overlap)}")

df1 duplicate tweet_ids: 0
df2 duplicate tweet_ids: 0
Overlap between df1 and df2: 868


In [26]:
print("df1 columns:", df1.columns.tolist())
print("df2 columns:", df2.columns.tolist())
print("df1 shape:", df1.shape)
print("df2 shape:", df2.shape)
print("\ndf1 head:")
print(df1.head(3))
print("\ndf2 head:")
print(df2.head(3))


df1 columns: ['tweet_id', 'sentiment', 'text', 'emotion']
df2 columns: ['tweet_id', 'label']
df1 shape: (2825, 4)
df2 shape: (4400, 2)

df1 head:
   tweet_id sentiment                                               text  \
0     28221  negative  @ DrDeng9 @ theskindoctor13 @ ANI @ MamataOffi...   
1       414  negative  @ waseemAli0007 @_ Mansoor _ Ali @ ImranKhanPT...   
2     29875  negative  @ UrmileshJ Kuchh nhi hoga media wale aur nich...   

  emotion  
0   anger  
1   anger  
2   anger  

df2 head:
   tweet_id  label
0     25800    joy
1     13738  anger
2     17016    joy


In [28]:
import pandas as pd

df1 = pd.read_csv("../data/emotion_labeled_1.csv")
df2 = pd.read_csv("../data/emotion_labeled_2.csv")

print("df1 shape:", df1.shape)
print("df1 columns:", df1.columns.tolist())
print("\ndf2 shape:", df2.shape)
print("df2 columns:", df2.columns.tolist())

df1 shape: (2825, 4)
df1 columns: ['tweet_id', 'sentiment', 'text', 'emotion']

df2 shape: (4400, 2)
df2 columns: ['tweet_id', 'label']


In [29]:
thontrain_custom = pd.read_csv("../data/train_custom.csv")

df2 = df2.merge(train_custom[['tweet_id', 'text']], on='tweet_id', how='left')
df2 = df2.rename(columns={'label': 'emotion'})

print("df2 shape after fix:", df2.shape)
print("df2 columns:", df2.columns.tolist())
print("Missing text:", df2['text'].isna().sum())

df2 shape after fix: (4400, 3)
df2 columns: ['tweet_id', 'emotion', 'text']
Missing text: 0


In [31]:
# Keep only what we need from df1
df1_clean = df1[['tweet_id', 'text', 'emotion']]
df2_clean = df2[['tweet_id', 'text', 'emotion']]

# Merge
merged = pd.concat([df1_clean, df2_clean], ignore_index=True)
print(f"Before dedup: {len(merged)}")

# Deduplicate
merged = merged.drop_duplicates(subset='tweet_id', keep='first')
print(f"After dedup: {len(merged)}")

print("\nEmotion distribution:")
print(merged['emotion'].value_counts())

Before dedup: 7225
After dedup: 6357

Emotion distribution:
emotion
anger        2794
joy          2341
sadness       502
trust         343
error         255
surprise      103
fear           18
confusion       1
Name: count, dtype: int64


In [32]:
# Keep only 4 emotion classes
keep = ['anger', 'joy', 'sadness', 'trust']
merged_clean = merged[merged['emotion'].isin(keep)].reset_index(drop=True)

print(merged_clean['emotion'].value_counts())
print(f"Total: {len(merged_clean)}")

# Save merged file
merged_clean.to_csv("../data/emotion_merged.csv", index=False)
print("✓ Saved emotion_merged.csv")

emotion
anger      2794
joy        2341
sadness     502
trust       343
Name: count, dtype: int64
Total: 5980
✓ Saved emotion_merged.csv


In [33]:
from sklearn.model_selection import train_test_split

train, dev = train_test_split(merged_clean, test_size=0.15,
                               stratify=merged_clean['emotion'],
                               random_state=42)

# Verify zero overlap
overlap = pd.merge(train, dev, on='tweet_id')
print(f"Overlap: {len(overlap)}")
print(f"Train: {len(train)}")
print(f"Dev: {len(dev)}")
print("\nTrain distribution:")
print(train['emotion'].value_counts())
print("\nDev distribution:")
print(dev['emotion'].value_counts())

Overlap: 0
Train: 5083
Dev: 897

Train distribution:
emotion
anger      2375
joy        1990
sadness     427
trust       291
Name: count, dtype: int64

Dev distribution:
emotion
anger      419
joy        351
sadness     75
trust       52
Name: count, dtype: int64


In [34]:
train.to_csv("../data/emotion_train_final.csv", index=False)
dev.to_csv("../data/emotion_dev_final.csv", index=False)
print("✓ Saved!")

✓ Saved!


In [35]:
train_check = pd.read_csv("../data/emotion_train_final.csv")
dev_check = pd.read_csv("../data/emotion_dev_final.csv")
overlap_check = pd.merge(train_check, dev_check, on='tweet_id')
print(f"Final overlap: {len(overlap_check)}")
print(f"Train: {len(train_check)}, Dev: {len(dev_check)}")

Final overlap: 0
Train: 5083, Dev: 897
